In [6]:

# Setup and data loading

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

LAST_NAME = "Rojas"
STUDENT_ID = "TUPM-26-1244"  # same real student ID as the main activity

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

WORK_DIR = Path.cwd()
OUTPUT_DIR = WORK_DIR  # outputs are saved in the additional-exercises folder

DATA_DIR = None
for folder in [WORK_DIR / "data", WORK_DIR.parent / "data"]:
    if (folder / "secom.data").is_file() and (folder / "secom_labels.data").is_file():
        DATA_DIR = folder
        break

assert DATA_DIR is not None, "secom.data / secom_labels.data not found in ./data or ../data"

raw = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)
labels = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)
assert raw.ndim == 2 and len(labels) == len(raw)

n_records, n_features = raw.shape
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]
record_ids = np.arange(n_records)  # original SECOM record identifiers
is_pass = labels == -1
is_fail = labels == 1

missing = np.isnan(raw)
observed_n = (~missing).sum(axis=0)
observed_min = np.where(missing, np.inf, raw).min(axis=0)
observed_max = np.where(missing, -np.inf, raw).max(axis=0)
usable_mask = (observed_n > 0) & (observed_min != observed_max)
usable_idx = np.flatnonzero(usable_mask)

print("Workspace:", WORK_DIR.resolve())
print("Data folder:", DATA_DIR.resolve())
print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Random Seed:", RANDOM_SEED)
print("Records:", n_records, "| Features:", n_features,
      "| Usable features:", len(usable_idx))


Workspace: C:\Users\rojas\Lab7_NumPy_Pandas_Rojas\Lab7_NumPy_Pandas_Additional_Exercises\Lab7_NumPy_Pandas_Additional_Exercises
Data folder: C:\Users\rojas\Lab7_NumPy_Pandas_Rojas\Lab7_NumPy_Pandas_Additional_Exercises\data
Student: Rojas TUPM-26-1244
Seed Digit: 4 | ID Sum: 19 | Random Seed: 261244
Records: 1567 | Features: 590 | Usable features: 474


In [7]:
# Exercise 1, Vectorized Measurement Quality Analysis

k = 5 + (SEED_DIGIT % 3)

rng1 = np.random.default_rng(RANDOM_SEED)
sel1 = usable_idx[rng1.choice(len(usable_idx), size=k, replace=False)]
names1 = [feature_names[i] for i in sel1]
X = raw[:, sel1]

means = np.nanmean(X, axis=0)
medians = np.nanmedian(X, axis=0)
sds = np.nanstd(X, axis=0, ddof=1)
missing_pct = np.isnan(X).mean(axis=0) * 100

# Robust standardization: median and MAD (0.6745 scaling = modified z-score)
mad = np.nanmedian(np.abs(X - medians), axis=0)
safe_mad = np.where(mad > 0, mad, np.nan)      # zero MAD -> no robust score
robust_z = 0.6745 * (X - medians) / safe_mad

with np.errstate(invalid="ignore"):
    flags = np.abs(robust_z) > 3.5             # NaN compares as False

observed_count = (~np.isnan(X)).sum(axis=0)
flagged_count = flags.sum(axis=0)
flagged_pct = 100 * flagged_count / observed_count

quality = pd.DataFrame({
    "Feature": names1,
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "Missing_Percent": missing_pct,
    "MAD": mad,
    "Flagged_Count": flagged_count,
    "Flagged_Percent": flagged_pct,
})
quality = quality.sort_values(
    ["Flagged_Percent", "Flagged_Count"], ascending=False, kind="stable"
).reset_index(drop=True)
quality.insert(0, "Rank", quality.index + 1)
quality.to_csv(OUTPUT_DIR / "measurement_quality.csv", index=False)

# Verify the vectorized results against Pandas and a plain loop
pdf = pd.DataFrame(X, columns=names1)
checks = [
    np.allclose(means, pdf.mean().to_numpy()),
    np.allclose(medians, pdf.median().to_numpy()),
    np.allclose(sds, pdf.std(ddof=1).to_numpy()),
    np.allclose(missing_pct, pdf.isna().mean().to_numpy() * 100),
]
loop_total = 0
for j in range(X.shape[1]):
    vals = X[:, j][~np.isnan(X[:, j])]
    med = np.median(vals)
    m = np.median(np.abs(vals - med))
    if m > 0:
        loop_total += int((np.abs(0.6745 * (vals - med) / m) > 3.5).sum())
checks.append(loop_total == int(flags.sum()))
verified1 = all(checks)

top_missing = quality.loc[quality["Missing_Percent"].idxmax()]
top_flagged = quality.iloc[0]

print(quality.round(4))
print("Selected Features:", names1)
print("Highest Missing Percentage:",
      f"{top_missing['Missing_Percent']:.2f}%", f"({top_missing['Feature']})")
print("Feature with Highest Flagged Percentage:",
      top_flagged["Feature"], f"({top_flagged['Flagged_Percent']:.2f}%)")
print("Total Flagged Observations:", int(flags.sum()))
print("Vectorized Results Verified:", verified1)
print("Saved: measurement_quality.csv")

   Rank      Feature       Mean     Median  Sample_SD  Missing_Percent  \
0     1  Feature_040    67.9049    78.2900    24.0629           1.5316   
1     2  Feature_577    16.6424    13.8179    12.4853           0.0000   
2     3  Feature_000  3014.4529  3011.4900    73.6218           0.3829   
3     4  Feature_208    73.2643    73.2480    28.0671           0.3829   
4     5  Feature_014     9.0054     8.9670     2.7966           0.1914   
5     6  Feature_539     3.0342     3.0548     1.2529           0.5105   

       MAD  Flagged_Count  Flagged_Percent  
0   2.1400            337          21.8406  
1   2.8725             78           4.9777  
2  45.2300             12           0.7687  
3  17.2120              6           0.3844  
4   1.8872              2           0.1279  
5   1.0167              0           0.0000  
Selected Features: ['Feature_014', 'Feature_208', 'Feature_040', 'Feature_577', 'Feature_000', 'Feature_539']
Highest Missing Percentage: 1.53% (Feature_040)
Feature 

In [8]:
# Exercise 2, Pass/Fail grouped comparison

full = pd.DataFrame(
    raw, columns=feature_names,
    index=pd.Index(record_ids, name="Example_ID"),
)
full["Result"] = labels
full["Outcome"] = np.where(labels == -1, "Pass", "Fail")

pass_count = int(is_pass.sum())
fail_count = int(is_fail.sum())
per_group = min(20 + ID_SUM, pass_count, fail_count)

rng2 = np.random.default_rng(RANDOM_SEED)
sel2 = usable_idx[rng2.choice(len(usable_idx), size=5, replace=False)]
names2 = [feature_names[i] for i in sel2]

pass_ids = rng2.choice(record_ids[is_pass], size=per_group, replace=False)
fail_ids = rng2.choice(record_ids[is_fail], size=per_group, replace=False)
sample_ids = np.sort(np.concatenate([pass_ids, fail_ids]))

balanced = full.loc[sample_ids, names2 + ["Result", "Outcome"]].copy()

full_medians = full[names2].median()            # medians from the FULL dataset
balanced[names2] = balanced[names2].fillna(full_medians)

grouped = balanced.groupby("Outcome")[names2].agg(
    ["count", "mean", "median", "std"]
)

rows = []
for f in names2:
    n_p, n_f = grouped.loc["Pass", (f, "count")], grouped.loc["Fail", (f, "count")]
    m_p, m_f = grouped.loc["Pass", (f, "mean")], grouped.loc["Fail", (f, "mean")]
    s_p, s_f = grouped.loc["Pass", (f, "std")], grouped.loc["Fail", (f, "std")]
    pooled_sd = np.sqrt(((n_p - 1) * s_p**2 + (n_f - 1) * s_f**2) / (n_p + n_f - 2))
    diff = m_f - m_p                              # Fail - Pass
    smd = diff / pooled_sd if np.isfinite(pooled_sd) and pooled_sd > 0 else 0.0
    rows.append({
        "Feature": f,
        "Pass_Count": int(n_p), "Fail_Count": int(n_f),
        "Pass_Mean": m_p, "Fail_Mean": m_f,
        "Pass_Median": grouped.loc["Pass", (f, "median")],
        "Fail_Median": grouped.loc["Fail", (f, "median")],
        "Pass_SD": s_p, "Fail_SD": s_f,
        "Abs_Mean_Difference": abs(diff),
        "Pooled_SD": pooled_sd,
        "Standardized_Mean_Difference": smd,
        "Abs_Standardized_Mean_Difference": abs(smd),
    })

comparison = pd.DataFrame(rows).sort_values(
    "Abs_Standardized_Mean_Difference", ascending=False
).reset_index(drop=True)
comparison.insert(0, "Rank", comparison.index + 1)

balanced.reset_index().to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

# NumPy vs Pandas check (sample SD uses ddof=1)
match2 = True
for f in names2:
    for g in ["Pass", "Fail"]:
        vals = balanced.loc[balanced["Outcome"] == g, f].to_numpy()
        match2 &= bool(np.isclose(vals.mean(), grouped.loc[g, (f, "mean")]))
        match2 &= bool(np.isclose(vals.std(ddof=1), grouped.loc[g, (f, "std")]))

print("Grouped Statistics:\n", grouped.round(4))
print(comparison.round(4))
print("Balanced Sample Size:", len(balanced), f"({per_group} per outcome)")
print("Selected Features:", names2)
print("Highest-Ranked Feature:", comparison.loc[0, "Feature"])
print("Absolute Standardized Mean Difference:",
      f"{comparison.loc[0, 'Abs_Standardized_Mean_Difference']:.4f}")
print("NumPy-Pandas Statistics Match:", match2)
print("Saved: balanced_sample.csv, feature_comparison.csv")

Grouped Statistics:
         Feature_000                              Feature_208                   \
              count       mean   median      std       count     mean  median   
Outcome                                                                         
Fail             39  2995.6238  2980.84  82.9816          39  72.3577  73.696   
Pass             39  3026.7603  3029.98  65.8585          39  79.1581  80.721   

                 Feature_014                         Feature_578          \
             std       count    mean  median     std       count    mean   
Outcome                                                                    
Fail     27.3343          39  8.2397  8.0652  2.8029          39  0.0202   
Pass     22.5186          39  8.5085  8.3811  2.9194          39  0.0209   

                        Feature_540                          
         median     std       count    mean  median     std  
Outcome                                                      
Fail  

In [9]:
# Exercise 3, Robustness and sensitivity

MIN_OBS = 10
B = 200 + 10 * SEED_DIGIT

pass_obs = (~np.isnan(raw[is_pass])).sum(axis=0)
fail_obs = (~np.isnan(raw[is_fail])).sum(axis=0)
eligible = np.flatnonzero(usable_mask & (pass_obs >= MIN_OBS) & (fail_obs >= MIN_OBS))

def select_features(seed):
    r = np.random.default_rng(seed)
    return eligible[r.choice(len(eligible), size=5, replace=False)]

sel3 = select_features(RANDOM_SEED)
names3 = [feature_names[i] for i in sel3]

Xp = raw[is_pass][:, sel3]
Xf = raw[is_fail][:, sel3]
full_med3 = np.nanmedian(raw[:, sel3], axis=0)          # full-dataset medians
Xp_imp = np.where(np.isnan(Xp), full_med3, Xp)
Xf_imp = np.where(np.isnan(Xf), full_med3, Xf)

# Observed Fail - Pass mean differences under both strategies
diff_avail = np.nanmean(Xf, axis=0) - np.nanmean(Xp, axis=0)
diff_imp = Xf_imp.mean(axis=0) - Xp_imp.mean(axis=0)

def stratified_bootstrap(seed, reps):
    r = np.random.default_rng(seed)
    idx_p = r.integers(0, len(Xp), size=(reps, len(Xp)))   # Pass group only
    idx_f = r.integers(0, len(Xf), size=(reps, len(Xf)))   # Fail group only
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        avail = np.nanmean(Xf[idx_f], axis=1) - np.nanmean(Xp[idx_p], axis=1)
    imp = Xf_imp[idx_f].mean(axis=1) - Xp_imp[idx_p].mean(axis=1)
    return avail, imp                                      # shape (reps, 5)

boot_avail, boot_imp = stratified_bootstrap(RANDOM_SEED, B)

def summarize(boot):
    return (np.nanmean(boot, axis=0),
            np.nanpercentile(boot, 2.5, axis=0),
            np.nanpercentile(boot, 97.5, axis=0))

bm_a, lo_a, hi_a = summarize(boot_avail)
bm_i, lo_i, hi_i = summarize(boot_imp)

def rank_abs(values):
    order = np.argsort(-np.abs(values), kind="stable")
    ranks = np.empty(len(values), dtype=int)
    ranks[order] = np.arange(1, len(values) + 1)
    return ranks

rank_a, rank_i = rank_abs(diff_avail), rank_abs(diff_imp)

change = np.abs(diff_imp - diff_avail)
sign_changed = np.sign(diff_avail) != np.sign(diff_imp)
rank_changed = rank_a != rank_i
excl_a = (lo_a > 0) | (hi_a < 0)
excl_i = (lo_i > 0) | (hi_i < 0)
ci_overlap = np.maximum(lo_a, lo_i) <= np.minimum(hi_a, hi_i)
big = int(np.argmax(change))

# Reproducibility: repeat feature selection and bootstrap with the same seed
again_a, again_i = stratified_bootstrap(RANDOM_SEED, B)
reproducible = bool(
    np.array_equal(select_features(RANDOM_SEED), sel3)
    and np.array_equal(boot_avail, again_a, equal_nan=True)
    and np.array_equal(boot_imp, again_i, equal_nan=True)
)

robustness = pd.DataFrame({
    "Feature": names3,
    "Pass_Observed": (~np.isnan(Xp)).sum(axis=0),
    "Fail_Observed": (~np.isnan(Xf)).sum(axis=0),
    "Diff_AvailableCase": diff_avail,
    "Diff_MedianImputed": diff_imp,
    "Change_In_Diff": change,
    "Sign_Changed": sign_changed,
    "Boot_Mean_AvailableCase": bm_a,
    "CI_Low_AvailableCase": lo_a,
    "CI_High_AvailableCase": hi_a,
    "CI_Excludes_Zero_AvailableCase": excl_a,
    "Boot_Mean_MedianImputed": bm_i,
    "CI_Low_MedianImputed": lo_i,
    "CI_High_MedianImputed": hi_i,
    "CI_Excludes_Zero_MedianImputed": excl_i,
    "CIs_Overlap": ci_overlap,
    "Rank_AvailableCase": rank_a,
    "Rank_MedianImputed": rank_i,
    "Rank_Changed": rank_changed,
})
robustness.to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)

n = len(names3)
report_lines = [
    "SECOM SENSITIVITY AND ROBUSTNESS REPORT",
    "---------------------------------------",
    f"Student: {LAST_NAME} {STUDENT_ID}",
    f"Random Seed: {RANDOM_SEED}",
    f"Selected Features: {', '.join(names3)}",
    f"Bootstrap Repetitions: {B}",
    f"Largest Change in Mean Difference: {names3[big]} ({change[big]:.4f})",
    f"Features with Changed Difference Sign: {int(sign_changed.sum())}",
    f"Features with Changed Ranking: {int(rank_changed.sum())}",
    f"95% CI excludes zero (available-case): {int(excl_a.sum())} of {n}",
    f"95% CI excludes zero (median-imputed): {int(excl_i.sum())} of {n}",
    f"Reproducibility Verified: {reproducible}",
    "",
    "Interpretation:",
    ("- The direction of every mean difference is the same under both strategies."
     if not sign_changed.any() else
     "- At least one mean difference changes direction, so the result is sensitive to missing-value handling."),
    ("- Feature rankings are the same under both strategies."
     if not rank_changed.any() else
     "- Feature rankings change between strategies, so the ordering depends on the preparation method."),
    "- A CI that includes zero means the Fail-Pass difference is not clearly separated from zero.",
    "- Median imputation uses full-dataset medians and can shrink differences when many values are missing.",
    "- Group differences describe association and do not establish failure causes.",
]
(OUTPUT_DIR / "sensitivity_report.txt").write_text(
    "\n".join(report_lines) + "\n", encoding="utf-8"
)

print(robustness.round(4).T)
print("Selected Features:", names3)
print("Bootstrap Repetitions:", B)
print("Largest Change in Mean Difference:",
      names3[big], f"{change[big]:.4f}")
print("Features with Changed Difference Sign:", int(sign_changed.sum()))
print("Features with Changed Ranking:", int(rank_changed.sum()))
print("Reproducibility Verified:", reproducible)
print("Saved: robustness_comparison.csv, sensitivity_report.txt")

                                          0            1            2  \
Feature                         Feature_000  Feature_208  Feature_014   
Pass_Observed                          1457         1457         1460   
Fail_Observed                           104          104          104   
Diff_AvailableCase                  -7.4211      -2.5615       -0.774   
Diff_MedianImputed                  -7.4069      -2.5608      -0.7738   
Change_In_Diff                       0.0142       0.0008       0.0002   
Sign_Changed                          False        False        False   
Boot_Mean_AvailableCase             -6.7622      -2.2013      -0.7774   
CI_Low_AvailableCase               -23.5313      -6.8276      -1.2581   
CI_High_AvailableCase               11.0779       3.5474      -0.2507   
CI_Excludes_Zero_AvailableCase        False        False         True   
Boot_Mean_MedianImputed             -6.7473       -2.201      -0.7772   
CI_Low_MedianImputed               -23.5164      -6